# 04 · Escrita concorrente no DuckDB

**Pergunta** (item 1 de [pontos-a-verificar.md](../../docs/arquitetura/2.0-lake-aberto/pontos-a-verificar.md)):
o DuckDB é **single-writer**. O que acontece quando vários conectores escrevem no
**mesmo banco** ao mesmo tempo — e como serializar?

**Como isso aparece na Strattum.** Cada conector é um **flow Prefect próprio**
(`airtable_sync`, `hubspot_sync`, …) — processo separado. No fim de cada flow, o
`dbt run` roda como **subprocess** ([flows/utils.py](../../strattum-data/services/pipelines/src/flows/utils.py) `_run_dbt`)
e escreve os modelos clean no **mesmo arquivo** `strattum.duckdb` (`DBT_DUCKDB_PATH`).

| Camada | Onde grava | Disputa de lock? |
|---|---|---|
| **RAW** | Delta Lake, um path por conector (`/data/raw/{conector}/`) | **não** — arquivos separados |
| **CLEAN** | `strattum.duckdb` **único** (dbt) | **sim** — dois `dbt run` no mesmo arquivo brigam pelo lock |

Como conector = **processo**, reproduzo o caso real com `subprocess` (sem threads — não
é o modelo da Strattum).

**Roteiro:** A) 1 writer (baseline) → B) N processos no mesmo arquivo (o caso real, e o
que quebra) → **Solução:** limite de concorrência = 1 no Prefect.

## Setup

In [1]:
import sys; sys.path.insert(0, "/Users/allanfraga/Repos/strattum/experimentacoes")
import exp, duckdb, os, subprocess, textwrap

print("duckdb", duckdb.__version__)

# banco de teste isolado nesta pasta de dados
DIR = f"{exp.DATA}/concurrency"
os.makedirs(DIR, exist_ok=True)
DB = f"{DIR}/concurrency.duckdb"

def reset(db=DB):
    """Apaga o arquivo do banco (e o WAL) pra cada experimento começar limpo."""
    for suffix in ("", ".wal"):
        p = db + suffix
        if os.path.exists(p):
            os.remove(p)

def rows(db=DB, table="t"):
    """Conta linhas num processo à parte (read-only, não disputa o lock)."""
    out = subprocess.run(
        [sys.executable, "-c",
         f"import duckdb; print(duckdb.connect({db!r}, read_only=True)"
         f".sql('SELECT count(*) FROM {table}').fetchone()[0])"],
        capture_output=True, text=True)
    return out.stdout.strip() or f"(erro: {(out.stderr.strip().splitlines() or [''])[-1][:60]})"

print("DB:", DB)

duckdb 1.5.4
DB: /Users/allanfraga/Repos/strattum/experimentacoes/.data/concurrency/concurrency.duckdb


## A · Baseline — 1 writer funciona

Sanidade: um único conector escrevendo. Sem concorrência, sem surpresa.

In [2]:
reset()
con = duckdb.connect(DB)                       # abre read-write
con.execute("CREATE TABLE t (cid INT, i INT)")
con.executemany("INSERT INTO t VALUES (?, ?)", [(0, i) for i in range(10_000)])
con.close()
print("linhas:", rows())                        # esperado: 10.000

linhas: 10000


## B · O caso real — N conectores (processos) no MESMO arquivo

Cada "conector" é um **processo separado** que abre o mesmo `strattum.duckdb` em
read-write e escreve — segurando a conexão alguns segundos pra **garantir sobreposição**
(é o que acontece quando dois flows terminam juntos e disparam `dbt run` no mesmo banco).

**O que observar:** o DuckDB só deixa **um** processo com o lock de escrita. Os demais
**não esperam** — falham na hora, já no `connect()`, com `IOException: Could not set lock`.

In [3]:
reset()

# script que um conector roda como processo próprio: abre rw, segura, insere, solta
WORKER = textwrap.dedent("""
    import duckdb, sys, time
    db, cid, hold = sys.argv[1], int(sys.argv[2]), float(sys.argv[3])
    try:
        con = duckdb.connect(db)                       # <- disputa o lock de escrita aqui
        con.execute("CREATE TABLE IF NOT EXISTS t (cid INT, i INT)")
        time.sleep(hold)                               # segura o lock pra garantir overlap
        con.executemany("INSERT INTO t VALUES (?, ?)", [(cid, i) for i in range(10_000)])
        con.close()
        print(f"cid={cid} OK")
    except Exception as e:
        print(f"cid={cid} FALHOU: {type(e).__name__}: {str(e).splitlines()[0][:80]}")
""")

N, HOLD = 4, 3.0
# dispara os 4 de uma vez (todos brigam pelo lock ao mesmo tempo)
procs = [subprocess.Popen([sys.executable, "-c", WORKER, DB, str(cid), str(HOLD)],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
         for cid in range(N)]
for p in procs:
    print(p.communicate()[0].strip())

print("\nlinhas totais:", rows(), "(10k por processo que deu OK)")

cid=0 FALHOU: IOException: IO Error: Could not set lock on file "/Users/allanfraga/Repos/strattum/experimen


cid=1 OK
cid=2 FALHOU: IOException: IO Error: Could not set lock on file "/Users/allanfraga/Repos/strattum/experimen
cid=3 FALHOU: IOException: IO Error: Could not set lock on file "/Users/allanfraga/Repos/strattum/experimen

linhas totais: 10000 (10k por processo que deu OK)


## Solução · limite de concorrência = 1 (nativo do Prefect)

O Prefect tem **global concurrency limits**: marca-se o passo de escrita (`dbt run`) com
limite = 1 e o Prefect **enfileira** os flows — só um escreve por vez. Extract e transform
seguem em paralelo; só a **escrita** serializa.

Simulo o limite rodando os **mesmos processos, um de cada vez** (que é literalmente o que
"limite = 1" faz: enfileira). Agora todos escrevem, sem crash.

In [4]:
reset()
con = duckdb.connect(DB); con.execute("CREATE TABLE t (cid INT, i INT)"); con.close()

# limite = 1 => um processo por vez. Espera cada um terminar antes de soltar o próximo.
for cid in range(4):
    p = subprocess.Popen([sys.executable, "-c", WORKER, DB, str(cid), "0"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.communicate()[0].strip())              # bloqueia => serializa a escrita

print("\nlinhas totais:", rows(), "(esperado: 40.000 = 4 x 10.000)")

cid=0 OK


cid=1 OK


cid=2 OK


cid=3 OK

linhas totais: 40000 (esperado: 40.000 = 4 x 10.000)


## No Prefect de verdade (opcional)

Pra confirmar no orquestrador real, instale o Prefect no `.venv` e rode a célula abaixo.
Ela ocupa a tag de escrita com limite = 1 e dispara os conectores concorrentes — o efeito
deve bater com a solução acima (todos escrevem, serializados).

> Não roda sem o Prefect instalado — a célula detecta e só imprime as instruções.

In [5]:
try:
    from prefect import flow, task
    from prefect.concurrency.sync import concurrency
    HAS_PREFECT = True
except ImportError:
    HAS_PREFECT = False
    print("Prefect não instalado. Para validar:")
    print("  cd experimentacoes && .venv/bin/pip install prefect")
    print("  # criar o limite (uma vez):")
    print("  .venv/bin/prefect gcl create duckdb-write --limit 1")

if HAS_PREFECT:
    reset()
    con = duckdb.connect(DB); con.execute("CREATE TABLE t (cid INT, i INT)"); con.close()

    @task
    def write_connector(cid: int):
        # 'duckdb-write' = limite criado com: prefect gcl create duckdb-write --limit 1
        with concurrency("duckdb-write", occupy=1):        # só um flow escreve por vez
            c = duckdb.connect(DB)
            c.executemany("INSERT INTO t VALUES (?, ?)", [(cid, i) for i in range(10_000)])
            c.close()

    @flow
    def ingest_all():
        for f in [write_connector.submit(cid) for cid in range(4)]:
            f.result()

    ingest_all()
    print("linhas:", rows(), "(esperado: 40.000)")

Prefect não instalado. Para validar:
  cd experimentacoes && .venv/bin/pip install prefect
  # criar o limite (uma vez):
  .venv/bin/prefect gcl create duckdb-write --limit 1


## Recomendação

- **Default: limite de concorrência = 1 na tag de escrita do Prefect.** Custo quase zero,
  nativo, e cobre o pior caso — os `dbt run` rodam em **processos** e, sem serializar,
  quebram no lock (exp. B). Escreve serial; extract/transform seguem concorrentes.
- **Alternativa (ingestão pesada): um DuckDB por conector.** Escrita **paralela de
  verdade**, zero disputa; junta-se na leitura com `ATTACH`. Mais complexo e perde o
  "banco único" — só vale se a serialização virar gargalo.
- **Nunca** deixar N processos abrirem o mesmo `.duckdb` em escrita sem serializar: o
  DuckDB **não enfileira**, ele **falha na hora** (exp. B) — vira crash de flow.

Realimenta o item 1 de [pontos-a-verificar.md](../../docs/arquitetura/2.0-lake-aberto/pontos-a-verificar.md).